# Bloque 2: Entrenamiento de modelos con MLflow Tracking

Entrena dos familias de modelos (Regresion Logistica y Random Forest) sobre el training set del Bloque 1. Registra cada intento en MLflow con parametros, metricas, firma del modelo y dataset. Compara ambos modelos con PR-AUC y Precision@Top-20%.

In [0]:
# Imports: MLflow para tracking, sklearn para modelos
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score, log_loss
import numpy as np
import pandas as pd

# Constantes: mismas tablas del Bloque 1
CATALOG = "andina_market_lhcl_prod"
FEATURES_SCHEMA = "features"
OBSERVATIONS_TABLE = f"{CATALOG}.{FEATURES_SCHEMA}.customer_observations"
FEATURES_TABLE = f"{CATALOG}.{FEATURES_SCHEMA}.customer_features"
EXPERIMENT_NAME = "/Users/cqjoseadolfo@gmail.com/databricks_ai_harness/repurchase_propensity_block2"

# Configurar experimento MLflow
mlflow.set_experiment(EXPERIMENT_NAME)

# Cargar training set desde Unity Catalog
# Junta observaciones (cliente + corte + etiqueta) con features (12 variables)
train_df = spark.sql(f"""
    SELECT 
        o.customer_id, o.cutoff_ts, o.repurchased_next_30d AS label,
        f.days_since_last_order, f.orders_count_30d, f.orders_count_90d,
        f.avg_ticket_value_90d, f.payment_failure_rate_90d,
        f.view_product_count_14d, f.search_count_14d, f.web_activity_count_14d,
        f.cart_abandonment_sessions_14d, f.favorite_product_sku,
        f.tenure_days, f.customer_segment
    FROM {OBSERVATIONS_TABLE} o
    JOIN {FEATURES_TABLE} f
      ON o.customer_id = f.customer_id AND o.cutoff_ts = f.feature_timestamp
""").toPandas()

train_df['cutoff_ts'] = pd.to_datetime(train_df['cutoff_ts'])

# EDA: distribucion de etiquetas
print(f"Filas totales: {len(train_df)}")
print(f"\nDistribucion de etiquetas:")
print(train_df['label'].value_counts(normalize=True).round(3))

# EDA: tabla de tipos de features
feature_types = []
for col in train_df.columns:
    if col in ('customer_id', 'cutoff_ts'):
        continue
    if col == 'label':
        feature_types.append({'feature': col, 'tipo': 'etiqueta binaria'})
    elif train_df[col].dtype in ['int64', 'int32', 'float64']:
        feature_types.append({'feature': col, 'tipo': 'numerica'})
    else:
        feature_types.append({'feature': col, 'tipo': 'categorica'})
display(pd.DataFrame(feature_types))

# EDA: correlacion de features numericas con la etiqueta
numeric_cols = [c for c in train_df.columns 
               if c not in ('customer_id', 'cutoff_ts', 'label', 'favorite_product_sku', 'customer_segment')
               and train_df[c].dtype in ['int64', 'int32', 'float64']]
correlations = train_df[numeric_cols + ['label']].corr()['label'].drop('label').sort_values(ascending=False)
print(f"\nCorrelacion con la etiqueta (features numericas):")
print(correlations.round(3))

In [0]:
# Split temporal: train (jul-oct 2025), validation (nov-dic 2025), test (ene-mar 2026)
# Nunca se baraja entre cortes: el modelo se entrena con datos anteriores y se prueba con posteriores
train = train_df[train_df['cutoff_ts'] < '2025-11-01']
val = train_df[(train_df['cutoff_ts'] >= '2025-11-01') & (train_df['cutoff_ts'] < '2026-01-01')]
test = train_df[train_df['cutoff_ts'] >= '2026-01-01']

print(f"Train: {len(train)} filas ({train['cutoff_ts'].min().date()} a {train['cutoff_ts'].max().date()})")
print(f"Validation: {len(val)} filas ({val['cutoff_ts'].min().date()} a {val['cutoff_ts'].max().date()})")
print(f"Test: {len(test)} filas ({test['cutoff_ts'].min().date()} a {test['cutoff_ts'].max().date()})")

# Columnas de features
NUMERIC_FEATURES = ['days_since_last_order', 'orders_count_30d', 'orders_count_90d',
                    'avg_ticket_value_90d', 'payment_failure_rate_90d',
                    'view_product_count_14d', 'search_count_14d', 'web_activity_count_14d',
                    'cart_abandonment_sessions_14d', 'tenure_days']
CATEGORICAL_FEATURES = ['favorite_product_sku', 'customer_segment']
FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train, y_train = train[FEATURE_COLUMNS], train['label']
X_val, y_val = val[FEATURE_COLUMNS], val['label']
X_test, y_test = test[FEATURE_COLUMNS], test['label']

# Pipeline de preprocessing (viaja con el modelo al loguearse en MLflow)
# Numericas: imputar con media + escalar
# Categoricas: imputar con moda + one-hot encoding
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]), NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore'))
    ]), CATEGORICAL_FEATURES)
])

print(f"\nFeatures numericas: {len(NUMERIC_FEATURES)}")
print(f"Features categoricas: {len(CATEGORICAL_FEATURES)}")

In [0]:
# Modelo 1: Regresion Logistica (familia lineal)
# MLflow nested runs: run padre agrupa todos los intentos, cada hiperparametro es un run hijo

param_grid_lr = [
    {'C': 0.1, 'penalty': 'l2'},
    {'C': 1.0, 'penalty': 'l2'},
    {'C': 10.0, 'penalty': 'l2'},
]

best_lr_model = None
best_lr_score = -1
best_lr_params = None

with mlflow.start_run(run_name="Logistic_Regression") as parent_run:
    mlflow.set_tags({"model_family": "linear", "block": "2"})

    for params in param_grid_lr:
        run_name = f"LR_C{params['C']}"
        with mlflow.start_run(nested=True, run_name=run_name) as child_run:
            # Pipeline = preprocessing + modelo (todo en uno)
            pipeline = Pipeline([
                ('preprocessor', preprocessor),
                ('classifier', LogisticRegression(
                    C=params['C'], penalty=params['penalty'],
                    max_iter=1000, random_state=42
                ))
            ])
            pipeline.fit(X_train, y_train)

            # Evaluar en validation
            y_val_proba = pipeline.predict_proba(X_val)[:, 1]
            val_pr_auc = average_precision_score(y_val, y_val_proba)

            # Loguear en MLflow: parametros, metrica, modelo con firma
            mlflow.log_params(params)
            mlflow.log_metric("val_pr_auc", val_pr_auc)

            signature = infer_signature(
                X_train.head(100),
                pipeline.predict_proba(X_train.head(100))[:, 1]
            )
            mlflow.sklearn.log_model(
                pipeline, "model",
                signature=signature,
                input_example=X_train.head(3)
            )

            print(f"  {run_name}: PR-AUC(val) = {val_pr_auc:.4f}")

            if val_pr_auc > best_lr_score:
                best_lr_score = val_pr_auc
                best_lr_model = pipeline
                best_lr_params = params

    mlflow.log_metric("best_val_pr_auc", best_lr_score)

print(f"\nMejor LR: PR-AUC(val) = {best_lr_score:.4f} con C={best_lr_params['C']}")

In [0]:
# Modelo 2: Random Forest (familia de arboles)
# MLflow nested runs: run padre agrupa todos los intentos

param_grid_rf = [
    {'n_estimators': 50, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 5},
    {'n_estimators': 50, 'max_depth': 10},
    {'n_estimators': 100, 'max_depth': 10},
]

best_rf_model = None
best_rf_score = -1
best_rf_params = None

with mlflow.start_run(run_name="Random_Forest") as parent_run:
    mlflow.set_tags({"model_family": "trees", "block": "2"})

    for params in param_grid_rf:
        run_name = f"RF_n{params['n_estimators']}_d{params['max_depth']}"
        with mlflow.start_run(nested=True, run_name=run_name) as child_run:
            pipeline = Pipeline([
                ('preprocessor', preprocessor),
                ('classifier', RandomForestClassifier(
                    n_estimators=params['n_estimators'],
                    max_depth=params['max_depth'],
                    random_state=42
                ))
            ])
            pipeline.fit(X_train, y_train)

            y_val_proba = pipeline.predict_proba(X_val)[:, 1]
            val_pr_auc = average_precision_score(y_val, y_val_proba)

            mlflow.log_params(params)
            mlflow.log_metric("val_pr_auc", val_pr_auc)

            signature = infer_signature(
                X_train.head(100),
                pipeline.predict_proba(X_train.head(100))[:, 1]
            )
            mlflow.sklearn.log_model(
                pipeline, "model",
                signature=signature,
                input_example=X_train.head(3)
            )

            print(f"  {run_name}: PR-AUC(val) = {val_pr_auc:.4f}")

            if val_pr_auc > best_rf_score:
                best_rf_score = val_pr_auc
                best_rf_model = pipeline
                best_rf_params = params

    mlflow.log_metric("best_val_pr_auc", best_rf_score)

print(f"\nMejor RF: PR-AUC(val) = {best_rf_score:.4f} con n_estimators={best_rf_params['n_estimators']}, max_depth={best_rf_params['max_depth']}")

In [0]:
# Evaluacion en test set con el mejor modelo de cada familia

# Precision@Top-20%: de los clientes que el modelo mas recomienda (top 20%), que % compro
def precision_at_top_20(y_true, y_proba):
    df = pd.DataFrame({'y_true': y_true, 'y_proba': y_proba})
    df = df.sort_values('y_proba', ascending=False)
    top_20_pct = int(len(df) * 0.2)
    return df.head(top_20_pct)['y_true'].mean()

# Lift@Top-20%: precision@top20% / proporcion global de positivos
def lift_at_top_20(y_true, y_proba):
    base_rate = y_true.mean()
    p20 = precision_at_top_20(y_true, y_proba)
    return p20 / base_rate if base_rate > 0 else 0

def evaluate_model(model, X_test, y_test, model_name):
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    metrics = {
        'PR-AUC': average_precision_score(y_test, y_proba),
        'Precision@Top-20%': precision_at_top_20(y_test.values, y_proba),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'Lift@Top-20%': lift_at_top_20(y_test.values, y_proba),
        'LogLoss': log_loss(y_test, y_proba),
    }
    print(f"\n{model_name}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    return metrics

lr_metrics = evaluate_model(best_lr_model, X_test, y_test, "Regresion Logistica")
rf_metrics = evaluate_model(best_rf_model, X_test, y_test, "Random Forest")

# Tabla comparativa
print("\n" + "="*60)
print("COMPARACION (mejor de cada familia en test set)")
print("="*60)
comparison = pd.DataFrame([lr_metrics, rf_metrics],
                          index=['Logistic Regression', 'Random Forest'])
display(comparison.round(4))

# Loguear el mejor modelo final en MLflow
best_model = best_lr_model if lr_metrics['PR-AUC'] > rf_metrics['PR-AUC'] else best_rf_model
best_name = "Logistic Regression" if lr_metrics['PR-AUC'] > rf_metrics['PR-AUC'] else "Random Forest"
best_metrics = lr_metrics if best_model == best_lr_model else rf_metrics

with mlflow.start_run(run_name=f"Best_Model_{best_name.replace(' ', '_')}"):
    mlflow.set_tags({"block": "2", "best_model": best_name})
    for k, v in best_metrics.items():
        safe_key = k.replace('@', '_at_').replace('-', '_').lower()
        mlflow.log_metric(f"test_{safe_key}", v)

    signature = infer_signature(
        X_train.head(100),
        best_model.predict_proba(X_train.head(100))[:, 1]
    )
    mlflow.sklearn.log_model(
        best_model, "best_model",
        signature=signature,
        input_example=X_train.head(3)
    )

print(f"\nMejor modelo: {best_name} (PR-AUC = {max(lr_metrics['PR-AUC'], rf_metrics['PR-AUC']):.4f})")

## Resumen del Bloque 2

**Lo que se hizo:**
1. Se cargo el training set desde Unity Catalog (tabla `customer_features` + `customer_observations`)
2. EDA: distribucion de etiquetas, tipos de features, correlaciones con la etiqueta
3. Split temporal: train (jul-oct 2025), validation (nov-dic 2025), test (ene-mar 2026)
4. Dos familias de modelos: Regresion Logistica (lineal) y Random Forest (arboles)
5. MLflow nested runs: cada modelo tiene un run padre y runs hijos por hiperparametros
6. Metricas en test: PR-AUC, Precision@Top-20%, F1, ROC-AUC, Lift, LogLoss
7. Se logueo en MLflow: parametros, metricas, tags, firma, input_example, modelo

**Evidencias a capturar:**
1. MLflow UI mostrando runs anidados (padre + hijos)
2. Tabla de metricas comparativa
3. Firma del modelo e input_example en MLflow
4. Run del mejor modelo con todas las metricas de test